# E5-lite — 조기 탐지 판정기를 게이트웨이에 올리면 누출과 대기가 줄어드는가

## 왜 하나

E2에서 문장 앞부분 학습 판정기(`student_prefix`)는 오프라인 평가에서 문장 끝 판정과 같은 재현율로 탐지를 17토큰에서 6.5토큰으로 앞당겼다.
이것이 **실제 서비스 흐름**에서도 이득이 되는지 확인한다. S3에서 문장 끝 판정 게이트웨이는 두 가지 비용이 있었다.
hold는 첫 글자가 447ms 늦었고, pass는 환각 문장이 판정 전에 약 126자 나갔다.

## 무엇을 하나

게이트웨이에 **앞부분 재판정(stride)** 기능을 추가했다(SPEC §5.1).
- pass: 문장 도중에도 s토큰마다 앞부분을 판정해, 경보를 더 일찍 보낸다.
- hold: 앞부분이 안전 판정될 때마다 그때까지의 본문을 먼저 내보낸다.

세 가지 판정기 구성을 같은 조건에서 비교한다. 가짜 LLM은 **GPT 토큰 단위**로, 첫 토큰 300ms, 토큰당 25ms로 RAGTruth dev 60개 응답을 재생한다.

| 구성 | 판정기 | 판정 시점 | 임계값 |
|---|---|---|---|
| **B** (기준선, S3와 같음) | student_vark | 문장 끝 | E2 서비스 조건 보정값 (멀쩡한 응답 표시 10%) |
| **E-end** | student_prefix | 문장 끝 | 〃 |
| **E-s4** | student_prefix | 4토큰마다 앞부분 + 문장 끝 | 〃 |

임계값은 E2 노트북이 **검증 분할**로 보정해 저장한 값(`e2/e2_summary.csv`)을 그대로 쓴다. 이 노트북에서 새로 고르지 않는다.

## 지표 (환각 발생 시점 기준, SSOT 결정)

- 발생 위치 θ = 응답의 첫 환각 구간 시작 글자
- **발생 후 누출**: 첫 경보가 나온 순간까지 사용자에게 나간 글자 중 θ 이후 글자 수 (탐지된 응답 기준 중앙값)
- **엄격 재현율**: 첫 경보가 θ를 포함한 텍스트를 판정한 것일 때만 탐지
- **대기**: hold의 첫 글자 추가 지연, 글자당 평균 지연 (off 대비)
- 멀쩡한 응답 표시율, 본문 보존율, 판정 시간

## 사전 정의 기준 (SSOT §6에 고정, 결과 보기 전)

- **C1 대기**: E-s4 hold의 첫 글자 추가 지연 ≤ B hold의 50%
- **C2 누출**: E-s4 pass의 발생 후 누출 중앙값(탐지된 응답) ≤ B pass의 50%
- **C3 탐지 유지**: E-s4 pass의 엄격 재현율 ≥ B pass − 0.05
- **C4 기본**: 모든 구성에서 본문 보존 100%, 판정 p95 ≤ 30ms
- 60개 응답 중 환각 응답은 약 25개라 재현율은 변동이 크다. C3은 큰 퇴보가 없는지만 확인한다.

> 런타임 **GPU(T4)**. 드라이브에 `selfproject/student_vark`, `student_prefix`, `e2/e2_summary.csv`가 필요하다. 약 35~40분.


## 1. 환경 준비

In [ ]:
!pip -q install pysbd pytest-asyncio openai fastapi uvicorn httpx sentence-transformers tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'
!mkdir -p gateway/hallugate gateway/scripts gateway/tests

In [ ]:
import os, shutil, json, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/selfproject'
for m in ['student_vark', 'student_prefix']:
    if not os.path.isdir(m):
        shutil.copytree(os.path.join(OUT_DIR, m), m)
e2s = pd.read_csv(os.path.join(OUT_DIR, 'e2', 'e2_summary.csv'))
svc = e2s[e2s.target_clean_flag == 0.10].set_index('scorer').thr
THR = {'B': float(svc['B1 문장 끝']), 'E-end': float(svc['E2 문장 끝']), 'E-s4': float(svc['E2 앞부분 (s=4)'])}
print('임계값 (E2 검증 보정, 멀쩡한 응답 표시 10%):', {k: round(v, 3) for k, v in THR.items()})

## 2. 게이트웨이 코드 (hallugate)

파일만 만든다. 저장소의 코드와 같다.

In [ ]:
%%writefile gateway/hallugate/__init__.py
"""hallugate — RAG 응답을 스트리밍 중에 문장 단위로 검증하는 OpenAI 호환 게이트웨이."""
from .app import create_app
from .config import Settings

__all__ = ['create_app', 'Settings']

In [ ]:
%%writefile gateway/hallugate/config.py
"""게이트웨이 설정. 환경 변수로 덮어쓸 수 있다."""
import os
from dataclasses import dataclass, field


def _env(name, default, cast=str):
    v = os.environ.get(name)
    return default if v is None else cast(v)


@dataclass
class Settings:
    # 실제 LLM API (OpenAI 호환). 로컬 테스트 때는 mock_upstream 주소.
    upstream_base_url: str = field(default_factory=lambda: _env('UPSTREAM_BASE_URL', 'https://api.openai.com/v1'))
    upstream_api_key: str = field(default_factory=lambda: _env('UPSTREAM_API_KEY', ''))
    upstream_timeout: float = field(default_factory=lambda: _env('UPSTREAM_TIMEOUT', 120.0, float))

    # 판정기 (S2-b에서 학습한 student_vark)
    judge_model_dir: str = field(default_factory=lambda: _env('JUDGE_MODEL_DIR', 'student_vark'))
    judge_embedder: str = field(default_factory=lambda: _env('JUDGE_EMBEDDER', 'sentence-transformers/all-MiniLM-L6-v2'))
    judge_k: int = field(default_factory=lambda: _env('JUDGE_K', 8, int))
    judge_max_len: int = field(default_factory=lambda: _env('JUDGE_MAX_LEN', 512, int))
    judge_chunk_words: int = field(default_factory=lambda: _env('JUDGE_CHUNK_WORDS', 40, int))
    # 판정기 출력(logit)이 이 값 이상이면 환각으로 본다. 0.0 = 확률 0.5. S4에서 보정 예정.
    judge_threshold: float = field(default_factory=lambda: _env('JUDGE_THRESHOLD', 0.0, float))
    # 문장 앞부분 재판정 간격(업스트림 조각 수). 0이면 문장 끝에서만 판정. 앞부분 학습 판정기(student_prefix)와 함께 쓴다.
    judge_stride: int = field(default_factory=lambda: _env('JUDGE_STRIDE', 0, int))
    judge_device: str = field(default_factory=lambda: _env('JUDGE_DEVICE', ''))

    # 기본 정책 (요청마다 verification 필드로 바꿀 수 있음)
    #   pass : 토큰을 즉시 내보내고 판정 결과를 뒤따라 보냄 (지연 0, 틀린 문장이 먼저 나갈 수 있음)
    #   hold : 문장 단위로 붙잡았다가 판정 후 내보냄 (틀린 문장 차단 가능, 문장 길이만큼 지연)
    #   off  : 검증 없이 그대로 전달
    default_mode: str = field(default_factory=lambda: _env('VERIFY_MODE', 'pass'))
    #   flag : 환각 판정 문장에 표시만 함
    #   cut  : 환각 판정 시 응답을 끊음 (finish_reason=content_filter)
    default_action: str = field(default_factory=lambda: _env('VERIFY_ACTION', 'flag'))

    metrics_path: str = field(default_factory=lambda: _env('METRICS_PATH', 'gateway_metrics.jsonl'))

In [ ]:
%%writefile gateway/hallugate/context.py
"""요청에서 근거 문서(RAG 컨텍스트)와 검증 옵션을 꺼낸다.

우선순위
1. 최상위 필드 `verification.context` (문자열 또는 문자열 리스트)
   - OpenAI SDK에서는 `extra_body={"verification": {...}}` 로 보낸다.
2. 메시지 본문 안의 `<context>...</context>` 태그 (RAG 서비스가 흔히 쓰는 형식)
3. 둘 다 없으면 검증하지 않는다 (mode=off).

`verification` 필드는 실제 LLM API가 모르는 필드이므로 전달 전에 제거한다.
"""
import copy
import re

CTX_TAG = re.compile(r'<context>(.*?)</context>', re.S | re.I)
MODES = {'pass', 'hold', 'off'}
ACTIONS = {'flag', 'cut'}


def _content_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):          # [{"type":"text","text":...}, ...]
        return '\n'.join(p.get('text', '') for p in content if isinstance(p, dict))
    return ''


def extract(body, default_mode='pass', default_action='flag', default_stride=0, with_stride=False):
    """반환: (upstream으로 보낼 body, reference 문자열 또는 None, mode, action[, stride])
    with_stride=True면 앞부분 재판정 간격(stride)도 함께 돌려준다."""
    body = copy.deepcopy(body)
    opts = body.pop('verification', None) or {}

    ref = opts.get('context')
    if isinstance(ref, list):
        ref = '\n\n'.join(str(x) for x in ref)
    if not ref:
        found = []
        for m in body.get('messages', []):
            found += CTX_TAG.findall(_content_text(m.get('content')))
        ref = '\n\n'.join(s.strip() for s in found) or None

    mode = opts.get('mode', default_mode)
    action = opts.get('action', default_action)
    if mode not in MODES:
        raise ValueError(f'verification.mode must be one of {sorted(MODES)}')
    if action not in ACTIONS:
        raise ValueError(f'verification.action must be one of {sorted(ACTIONS)}')
    stride = opts.get('stride', default_stride)
    if not isinstance(stride, int) or isinstance(stride, bool) or stride < 0:
        raise ValueError('verification.stride must be a non-negative integer')
    if not ref:
        mode = 'off'
    return (body, ref, mode, action, stride) if with_stride else (body, ref, mode, action)

In [ ]:
%%writefile gateway/hallugate/splitter.py
"""스트리밍 텍스트를 문장 단위로 끊는다.

토큰이 조금씩 들어오므로, 버퍼에 쌓다가 '다음 문장이 시작된 것이 확인되면'
앞 문장을 완성된 것으로 보고 내보낸다. 마지막 문장은 스트림이 끝날 때 flush로 내보낸다.

내보낸 문장들을 이어 붙이면 원문과 정확히 같다(공백 포함). hold 모드에서 문장을
그대로 사용자에게 내보내야 하므로 이 성질이 중요하다.
"""
from dataclasses import dataclass

import pysbd


@dataclass
class Sentence:
    idx: int
    text: str        # 원문 그대로 (뒤 공백 포함)
    start: int       # 응답 전체에서의 시작 위치 (문자)

    @property
    def clean(self):
        return self.text.strip()


class IncrementalSplitter:
    def __init__(self, language='en'):
        self._seg = pysbd.Segmenter(language=language, clean=False, char_span=True)
        self._buf = ''
        self._offset = 0      # 버퍼 시작이 응답 전체에서 몇 번째 문자인지
        self._n = 0

    def _emit(self, text):
        s = Sentence(self._n, text, self._offset)
        self._n += 1
        self._offset += len(text)
        return s

    def feed(self, delta):
        """새 조각을 넣고, 완성된 문장 리스트를 돌려준다."""
        self._buf += delta
        if not self._buf.strip():
            return []
        starts = [sp.start for sp in self._seg.segment(self._buf) if sp.sent.strip()]
        if len(starts) < 2:
            return []
        out = []
        cuts = starts[1:]                 # 각 문장의 끝 = 다음 문장의 시작
        prev = 0
        for c in cuts:
            out.append(self._emit(self._buf[prev:c]))
            prev = c
        self._buf = self._buf[prev:]
        return out

    def pending(self):
        """아직 끝나지 않은 현재 문장: (응답 전체에서의 시작 위치, 지금까지의 텍스트)."""
        return self._offset, self._buf

    def flush(self):
        """스트림 종료 시 남은 텍스트를 마지막 문장으로 내보낸다 (공백뿐이면 판정은 건너뛰게 됨)."""
        if not self._buf:
            return []
        s = self._emit(self._buf)
        self._buf = ''
        return [s]

In [ ]:
%%writefile gateway/hallugate/judge.py
"""문장 판정기.

S2/S2-b와 같은 방식:
  근거 문서 → 40단어 조각 → 조각 임베딩 (요청당 1회, prepare)
  문장 → 임베딩 → 유사한 조각 top-k (원문 순서로 정렬) → 교차 인코더 → logit (클수록 환각)
"""
import threading
import time
from dataclasses import dataclass

import torch


@dataclass
class Verdict:
    score: float          # logit, 클수록 환각
    hallucinated: bool
    judge_ms: float       # 검색 + 판정 시간


@dataclass
class RefIndex:
    chunks: list
    emb: object           # torch.Tensor [n_chunks, d]
    prepare_ms: float


def chunk_text(text, n):
    w = text.split()
    return [' '.join(w[i:i + n]) for i in range(0, max(len(w), 1), n)] or [text]


class StudentJudge:
    def __init__(self, model_dir, embedder='sentence-transformers/all-MiniLM-L6-v2', k=8, max_len=512,
                 chunk_words=40, threshold=0.0, device=''):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(self.device).eval()
        if isinstance(embedder, str):
            from sentence_transformers import SentenceTransformer
            embedder = SentenceTransformer(embedder, device=self.device)
        self.embedder = embedder
        self.k, self.max_len, self.chunk_words, self.threshold = k, max_len, chunk_words, threshold
        self._lock = threading.Lock()     # 모델 1개를 여러 요청이 공유하므로 직렬화

    def _embed(self, texts):
        return self.embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True,
                                    show_progress_bar=False)

    def prepare(self, reference):
        t0 = time.perf_counter()
        chunks = chunk_text(reference, self.chunk_words)
        with self._lock:
            emb = self._embed(chunks)
        return RefIndex(chunks, emb, (time.perf_counter() - t0) * 1000)

    @torch.no_grad()
    def judge(self, ref, sentence):
        t0 = time.perf_counter()
        with self._lock:
            e = self._embed([sentence])[0]
            top = torch.topk(ref.emb @ e, min(self.k, ref.emb.shape[0])).indices.tolist()
            ctx = ' ... '.join(ref.chunks[i] for i in sorted(top))
            enc = self.tok([ctx], [sentence], truncation='only_first', max_length=self.max_len,
                           return_tensors='pt').to(self.device)
            with torch.autocast(self.device, enabled=(self.device == 'cuda')):
                score = float(self.model(**enc).logits.squeeze().float().cpu())
            if self.device == 'cuda':
                torch.cuda.synchronize()
        return Verdict(score, score >= self.threshold, (time.perf_counter() - t0) * 1000)


class LexicalJudge:
    """모델 없이 파이프라인을 시험하기 위한 대체 판정기 (테스트·데모 전용).
    근거에 없는 단어 비율을 점수로 쓴다. 판별력은 없다고 봐야 한다."""

    def __init__(self, threshold=0.5, delay_ms=0.0):
        self.threshold, self.delay_ms = threshold, delay_ms

    def prepare(self, reference):
        return RefIndex([reference], {w.lower().strip('.,') for w in reference.split()}, 0.0)

    def judge(self, ref, sentence):
        t0 = time.perf_counter()
        if self.delay_ms:
            time.sleep(self.delay_ms / 1000)
        words = [w.lower().strip('.,') for w in sentence.split() if w.strip('.,')]
        score = sum(w not in ref.emb for w in words) / max(len(words), 1)
        return Verdict(score, score >= self.threshold, (time.perf_counter() - t0) * 1000)

In [ ]:
%%writefile gateway/hallugate/pipeline.py
"""스트리밍 검증 파이프라인 (게이트웨이의 핵심).

입력: 실제 LLM이 흘려보내는 텍스트 조각(delta)
출력: 사용자에게 보낼 이벤트 — text(본문), verdict(판정 결과), cut(응답 중단)

정책 (SPEC §5)
  pass : 본문을 받는 즉시 내보내고, 판정은 백그라운드로 실행해 결과를 뒤따라 보낸다.
         추가 지연은 0이지만, 판정이 끝나기 전에 틀린 내용이 사용자에게 이미 나간다(누출).
  hold : 판정이 끝날 때까지 본문을 붙잡았다가 내보낸다. 누출은 0이지만 사용자는 기다려야 한다.
  action=cut 이면 환각 판정 시 그 자리에서 응답을 끊는다.

판정 시점 (SPEC §5.1)
  stride=0 : 문장이 완성되면 그 문장을 판정한다.
  stride=s : 위에 더해, 현재 문장에 조각이 s개 쌓일 때마다 '지금까지의 문장 앞부분'을 판정한다.
             pass는 문장 도중에도 경보를 낼 수 있고, hold는 안전 판정된 앞부분까지 먼저 내보낸다.

측정 (RequestRecord)
  sentences : 완성 문장 판정 — 완성 시각, 판정 시각, 전달 시각, 점수, 판정 시간
  checks    : 앞부분 판정 — 판정 구간, 시각, 점수, 판정 시점까지 사용자에게 나간 글자 수
"""
import asyncio
import time
from dataclasses import asdict, dataclass, field

from .splitter import IncrementalSplitter


@dataclass
class Event:
    kind: str                 # 'text' | 'verdict' | 'cut'
    text: str = ''
    info: dict = None


@dataclass
class RequestRecord:
    mode: str
    action: str
    stride: int = 0
    tag: str = ''                      # 요청 헤더 x-request-tag (측정용 식별자)
    t0: float = field(default_factory=time.perf_counter)
    prepare_ms: float = 0.0
    t_upstream_first: float = None     # 실제 LLM의 첫 토큰 도착 (ms, 요청 기준)
    t_client_first: float = None       # 사용자에게 첫 본문이 나간 시각
    t_upstream_done: float = None
    t_client_done: float = None
    released_chars: int = 0
    cut: bool = False
    sentences: list = field(default_factory=list)
    checks: list = field(default_factory=list)

    def now(self):
        return (time.perf_counter() - self.t0) * 1000

    def to_dict(self):
        d = asdict(self)
        d.pop('t0')
        return d


class VerifyPipeline:
    def __init__(self, judge, ref_index, mode, action, record, stride=0):
        assert mode in ('pass', 'hold')
        self.judge, self.ref, self.mode, self.action, self.rec = judge, ref_index, mode, action, record
        self.stride = stride or 0
        self.splitter = IncrementalSplitter()
        self.text = ''            # 지금까지 받은 전체 본문
        self.released_upto = 0    # hold: 사용자에게 내보낸 위치
        self.since = 0            # 현재 문장에 쌓인 조각 수

    # ---- 내부 도우미 ----
    def _release_text(self, text):
        if text:
            if self.rec.t_client_first is None:
                self.rec.t_client_first = self.rec.now()
            self.rec.released_chars += len(text)
        return Event('text', text)

    def _release_to(self, end):
        """hold: 본문을 end 위치까지 내보낸다."""
        piece = self.text[self.released_upto:end]
        self.released_upto = max(self.released_upto, end)
        return self._release_text(piece)

    async def _judge_sentence(self, s, srec):
        if not s.clean:                                   # 공백뿐인 꼬리
            srec.update(skipped=True, t_verdict=self.rec.now(), hallucinated=False)
            return None
        v = await asyncio.to_thread(self.judge.judge, self.ref, s.clean)
        srec.update(score=v.score, hallucinated=v.hallucinated, judge_ms=v.judge_ms,
                    t_verdict=self.rec.now(),
                    leak_chars=max(0, self.rec.released_chars - s.start) if v.hallucinated else 0)
        return {'kind': 'sentence', 'sentence_idx': s.idx, 'start': s.start, 'upto': s.start + len(s.text),
                'text': s.clean, 'score': v.score, 'hallucinated': v.hallucinated, 'judge_ms': v.judge_ms}

    async def _judge_prefix(self, sidx, start, raw):
        clean = raw.strip()
        if not clean:
            return None
        t_check = self.rec.now()
        v = await asyncio.to_thread(self.judge.judge, self.ref, clean)
        info = {'kind': 'prefix', 'sentence_idx': sidx, 'start': start, 'upto': start + len(raw), 'text': clean,
                'score': v.score, 'hallucinated': v.hallucinated, 'judge_ms': v.judge_ms}
        self.rec.checks.append({k: info[k] for k in ('sentence_idx', 'start', 'upto', 'score', 'hallucinated', 'judge_ms')}
                               | {'t_check': t_check, 't_verdict': self.rec.now()})
        return info

    def _verdict(self, info):
        info = dict(info, released_at_verdict=self.rec.released_chars)
        if info['kind'] == 'prefix':
            self.rec.checks[-1]['released_at_verdict'] = info['released_at_verdict']
        return Event('verdict', info=info)

    def _new_srec(self, s):
        srec = {'sentence_idx': s.idx, 'start': s.start, 'n_chars': len(s.text), 't_complete': self.rec.now()}
        self.rec.sentences.append(srec)
        return srec

    def _prefix_due(self, d, sents):
        """이번 조각으로 앞부분 재판정 시점이 됐는지. 문장이 끝났으면 새 문장 기준으로 다시 센다."""
        if d is None:
            return False
        if sents:
            _, buf = self.splitter.pending()
            self.since = 1 if buf.strip() else 0
        else:
            self.since += 1
        return bool(self.stride) and self.since > 0 and self.since % self.stride == 0

    def _cut(self, info):
        self.rec.cut = True
        return Event('cut', info={'sentence_idx': info['sentence_idx'], 'kind': info['kind']})

    # ---- 본체 ----
    async def run(self, deltas):
        q = asyncio.Queue()

        async def producer():
            try:
                async for d in deltas:
                    if self.rec.t_upstream_first is None:
                        self.rec.t_upstream_first = self.rec.now()
                    await q.put(d)
            finally:
                self.rec.t_upstream_done = self.rec.now()
                await q.put(None)

        prod = asyncio.create_task(producer())
        pending = []                                     # pass 모드: 판정 task들 (발생 순서)
        get = None
        try:
            while True:
                if get is None:
                    get = asyncio.create_task(q.get())
                # pass: 새 조각 도착 또는 가장 앞선 판정 완료 중 먼저 일어나는 쪽을 처리
                waits = {get} | ({pending[0]} if pending else set())
                done, _ = await asyncio.wait(waits, return_when=asyncio.FIRST_COMPLETED)

                while pending and pending[0].done():         # 끝난 판정은 순서대로 즉시 전달
                    info = pending.pop(0).result()
                    if info is None:
                        continue
                    yield self._verdict(info)
                    if info['hallucinated'] and self.action == 'cut':
                        yield self._cut(info)
                        return
                if get not in done:
                    continue
                d, get = get.result(), None
                if d:
                    self.text += d
                sents = self.splitter.flush() if d is None else self.splitter.feed(d)
                due = self._prefix_due(d, sents)

                if self.mode == 'pass':
                    if d:
                        yield self._release_text(d)
                    for s in sents:
                        srec = self._new_srec(s)
                        srec['t_release'] = srec['t_complete']   # 이미 나갔음
                        pending.append(asyncio.create_task(self._judge_sentence(s, srec)))
                    if due:
                        off, buf = self.splitter.pending()
                        pending.append(asyncio.create_task(self._judge_prefix(self.splitter._n, off, buf)))
                    if d is None:                                 # 스트림 종료: 남은 판정을 순서대로 기다림
                        for t in pending:
                            info = await t
                            if info is None:
                                continue
                            yield self._verdict(info)
                            if info['hallucinated'] and self.action == 'cut':
                                yield self._cut(info)
                                return
                        pending = []
                else:  # hold
                    for s in sents:
                        srec = self._new_srec(s)
                        info = await self._judge_sentence(s, srec)
                        if info is not None:
                            yield self._verdict(info)
                            if info['hallucinated'] and self.action == 'cut':
                                yield self._cut(info)
                                return
                        srec['t_release'] = self.rec.now()
                        yield self._release_to(s.start + len(s.text))
                    if due:
                        off, buf = self.splitter.pending()
                        info = await self._judge_prefix(self.splitter._n, off, buf)
                        if info is not None:
                            yield self._verdict(info)
                            if info['hallucinated'] and self.action == 'cut':
                                yield self._cut(info)
                                return
                            # 안전하거나(flag 모드에서는 표시 후) → 판정한 앞부분까지 내보낸다
                            yield self._release_to(off + len(buf))

                if d is None:
                    break
        finally:
            self.rec.t_client_done = self.rec.now()
            for t in pending:
                t.cancel()
            if get is not None:
                get.cancel()
            prod.cancel()
            try:
                await prod
            except (asyncio.CancelledError, Exception):
                pass

In [ ]:
%%writefile gateway/hallugate/app.py
"""OpenAI 호환 검증 게이트웨이.

서비스는 base_url만 이 게이트웨이로 바꾸면 된다.
    client = OpenAI(base_url="http://localhost:8000/v1", api_key=...)
    client.chat.completions.create(..., stream=True,
        extra_body={"verification": {"context": 근거문서, "mode": "hold", "action": "cut"}})

스트리밍 응답에는 일반 OpenAI 청크 사이에 판정 결과 청크가 섞여 나간다.
    {"object": "chat.completion.chunk", "choices": [], "verification": {...}}
choices가 비어 있으므로 일반 클라이언트는 본문에 영향 없이 무시하고, 필요한 클라이언트만
`chunk.verification` 으로 읽는다. 끊기면 finish_reason="content_filter".
"""
import asyncio
import json
import time
import uuid
from contextlib import asynccontextmanager

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, Response, StreamingResponse

from . import context as ctxmod
from .config import Settings
from .pipeline import RequestRecord, VerifyPipeline
from .splitter import IncrementalSplitter


def _sse(obj):
    return f'data: {json.dumps(obj, ensure_ascii=False)}\n\n'


async def _upstream_deltas(resp, state):
    """업스트림 SSE → 본문 텍스트 조각. 마지막 finish_reason과 청크 메타데이터를 state에 남긴다."""
    async for line in resp.aiter_lines():
        if not line.startswith('data:'):
            continue
        data = line[5:].strip()
        if data == '[DONE]':
            break
        chunk = json.loads(data)
        state.setdefault('id', chunk.get('id'))
        state.setdefault('model', chunk.get('model'))
        for ch in chunk.get('choices') or []:
            if ch.get('finish_reason'):
                state['finish_reason'] = ch['finish_reason']
            text = (ch.get('delta') or {}).get('content')
            if text:
                yield text


def create_app(settings=None, judge=None, http_client=None):
    settings = settings or Settings()
    state = {'judge': judge, 'client': http_client}

    @asynccontextmanager
    async def lifespan(app):
        if state['judge'] is None:
            from .judge import StudentJudge
            state['judge'] = StudentJudge(settings.judge_model_dir, settings.judge_embedder, settings.judge_k,
                                          settings.judge_max_len, settings.judge_chunk_words,
                                          settings.judge_threshold, settings.judge_device)
        # 예열: 첫 요청에서 GPU 커널 준비로 판정이 수백 ms 걸리는 것을 막는다 (S3 측정에서 첫 요청 468ms 관측)
        warm = state['judge'].prepare('warm up reference text ' * 40)
        for _ in range(3):
            state['judge'].judge(warm, 'This is a warm up sentence.')
        own_client = state['client'] is None
        if own_client:
            state['client'] = httpx.AsyncClient(timeout=settings.upstream_timeout)
        yield
        if own_client:
            await state['client'].aclose()

    app = FastAPI(title='hallugate', lifespan=lifespan)

    def _log(rec):
        if settings.metrics_path:
            with open(settings.metrics_path, 'a') as f:
                f.write(json.dumps(rec.to_dict(), ensure_ascii=False) + '\n')

    def _headers(request):
        h = {'Content-Type': 'application/json'}
        auth = f'Bearer {settings.upstream_api_key}' if settings.upstream_api_key else request.headers.get('authorization')
        if auth:
            h['Authorization'] = auth
        return h

    @app.get('/health')
    async def health():
        return {'ok': True, 'judge': type(state['judge']).__name__,
                'mode': settings.default_mode, 'action': settings.default_action}

    @app.post('/v1/chat/completions')
    async def chat(request: Request):
        raw = await request.json()
        try:
            body, ref, mode, action, stride = ctxmod.extract(raw, settings.default_mode, settings.default_action,
                                                             settings.judge_stride, with_stride=True)
        except ValueError as e:
            return JSONResponse({'error': {'message': str(e), 'type': 'invalid_request_error'}}, status_code=400)

        client, judge = state['client'], state['judge']
        url = settings.upstream_base_url.rstrip('/') + '/chat/completions'
        headers = _headers(request)
        rec = RequestRecord(mode=mode, action=action, stride=stride, tag=request.headers.get('x-request-tag', ''))
        vh = {'x-verification-mode': mode, 'x-verification-action': action, 'x-verification-stride': str(stride)}

        # ---------- 비스트리밍 ----------
        if not body.get('stream'):
            r = await client.post(url, json=body, headers=headers)
            if r.status_code != 200 or mode == 'off':
                return Response(r.content, status_code=r.status_code, media_type='application/json', headers=vh)
            data = r.json()
            content = data['choices'][0]['message'].get('content') or ''
            ref_idx = await asyncio.to_thread(judge.prepare, ref)
            rec.prepare_ms = ref_idx.prepare_ms
            sp = IncrementalSplitter()
            sents = sp.feed(content) + sp.flush()
            results, kept = [], ''
            for s in sents:
                if not s.clean:
                    kept += s.text
                    continue
                v = await asyncio.to_thread(judge.judge, ref_idx, s.clean)
                results.append({'sentence_idx': s.idx, 'start': s.start, 'text': s.clean, 'score': v.score,
                                'hallucinated': v.hallucinated, 'judge_ms': v.judge_ms})
                rec.sentences.append(dict(results[-1]))
                if v.hallucinated and action == 'cut':
                    rec.cut = True
                    break
                kept += s.text
            if rec.cut and mode == 'hold':            # hold+cut: 환각 문장부터 잘라서 반환
                data['choices'][0]['message']['content'] = kept
                data['choices'][0]['finish_reason'] = 'content_filter'
            data['verification'] = {'mode': mode, 'action': action, 'prepare_ms': ref_idx.prepare_ms,
                                    'sentences': results, 'cut': rec.cut}
            rec.t_client_done = rec.now()
            _log(rec)
            return JSONResponse(data, headers=vh)

        # ---------- 스트리밍 ----------
        req = client.build_request('POST', url, json=body, headers=headers)
        resp = await client.send(req, stream=True)
        if resp.status_code != 200:
            err = await resp.aread()
            await resp.aclose()
            return Response(err, status_code=resp.status_code, media_type='application/json', headers=vh)

        if mode == 'off':
            async def passthrough():
                try:
                    async for chunk in resp.aiter_raw():
                        yield chunk
                finally:
                    await resp.aclose()
            return StreamingResponse(passthrough(), media_type='text/event-stream', headers=vh)

        async def gen():
            meta = {}
            cid = 'chatcmpl-' + uuid.uuid4().hex[:24]
            created = int(time.time())

            def chunk(delta=None, finish=None, verification=None):
                c = {'id': meta.get('id') or cid, 'object': 'chat.completion.chunk', 'created': created,
                     'model': meta.get('model') or body.get('model', ''),
                     'choices': [] if delta is None and finish is None else
                     [{'index': 0, 'delta': delta or {}, 'finish_reason': finish}]}
                if verification is not None:
                    c['verification'] = verification
                return _sse(c)

            try:
                ref_idx = await asyncio.to_thread(judge.prepare, ref)
                rec.prepare_ms = ref_idx.prepare_ms
                yield chunk(delta={'role': 'assistant', 'content': ''})
                pipe = VerifyPipeline(judge, ref_idx, mode, action, rec, stride=stride)
                async for ev in pipe.run(_upstream_deltas(resp, meta)):
                    if ev.kind == 'text':
                        if ev.text:
                            yield chunk(delta={'content': ev.text})
                    elif ev.kind == 'verdict':
                        yield chunk(verification=ev.info)
                    elif ev.kind == 'cut':
                        yield chunk(finish='content_filter', verification={'cut': True, **ev.info})
                if not rec.cut:
                    yield chunk(finish=meta.get('finish_reason', 'stop'))
                yield 'data: [DONE]\n\n'
            finally:
                await resp.aclose()
                _log(rec)

        return StreamingResponse(gen(), media_type='text/event-stream', headers=vh)

    return app

In [ ]:
%%writefile gateway/serve.py
"""게이트웨이 실행.

  JUDGE_MODEL_DIR=student_vark UPSTREAM_BASE_URL=http://localhost:9000/v1 python serve.py --port 8000
  (판정기 없이 파이프라인만 시험: --lexical)
"""
import argparse

import uvicorn

from hallugate import Settings, create_app
from hallugate.judge import LexicalJudge

p = argparse.ArgumentParser()
p.add_argument('--port', type=int, default=8000)
p.add_argument('--host', default='127.0.0.1')
p.add_argument('--lexical', action='store_true', help='모델 없이 어휘 기반 대체 판정기 사용 (시험용)')
a = p.parse_args()

uvicorn.run(create_app(Settings(), judge=LexicalJudge() if a.lexical else None),
            host=a.host, port=a.port, log_level='warning')

In [ ]:
%%writefile gateway/mock_upstream.py
"""API 키 없이 게이트웨이를 시험하기 위한 가짜 LLM (OpenAI 호환).

요청 body에 다음 필드를 넣으면 그대로 재생한다 (실제 API로는 보내지 말 것):
  mock_response : 흘려보낼 답변 텍스트 (없으면 마지막 user 메시지를 되돌려줌)
  mock_ttft_ms  : 첫 토큰까지 지연 (기본 300ms)
  mock_token_ms : 토큰 간 간격 (기본 25ms ≈ 초당 40토큰, 상용 API와 비슷한 속도)

토큰 단위: 기본은 단어. 환경 변수 MOCK_TOKENIZER=tiktoken 이면 GPT 토큰(cl100k_base) 단위.

실행: python mock_upstream.py --port 9000
"""
import argparse
import asyncio
import json
import os
import re
import time
import uuid

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse

app = FastAPI(title='mock-llm')


_ENC = None
if os.environ.get('MOCK_TOKENIZER', 'words') == 'tiktoken':
    # 상용 API처럼 GPT 토큰 단위로 흘려보낸다 (E1·E2의 토큰 단위와 일치). 이어 붙이면 원문과 같다.
    import tiktoken
    _ENC = tiktoken.get_encoding('cl100k_base')


def _tokens(text):
    if _ENC is not None:
        ids = _ENC.encode(text)
        dec, starts = _ENC.decode_with_offsets(ids)
        if dec == text:
            ends = starts[1:] + [len(text)]
            return [text[a:b] for a, b in zip(starts, ends) if b > a]
    return re.findall(r'\S+\s*|\s+', text)


@app.post('/v1/chat/completions')
async def chat(request: Request):
    b = await request.json()
    text = b.get('mock_response')
    if text is None:
        users = [m for m in b.get('messages', []) if m.get('role') == 'user']
        text = users[-1]['content'] if users else 'Hello.'
    ttft, gap = b.get('mock_ttft_ms', 300) / 1000, b.get('mock_token_ms', 25) / 1000
    cid, created, model = 'chatcmpl-mock' + uuid.uuid4().hex[:12], int(time.time()), b.get('model', 'mock')

    if not b.get('stream'):
        await asyncio.sleep(ttft + gap * len(_tokens(text)))
        return JSONResponse({'id': cid, 'object': 'chat.completion', 'created': created, 'model': model,
                             'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': text},
                                          'finish_reason': 'stop'}]})

    def c(delta, finish=None):
        return 'data: ' + json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': created,
                                      'model': model, 'choices': [{'index': 0, 'delta': delta,
                                                                   'finish_reason': finish}]}) + '\n\n'

    async def gen():
        yield c({'role': 'assistant', 'content': ''})
        await asyncio.sleep(ttft)
        for tok in _tokens(text):
            yield c({'content': tok})
            await asyncio.sleep(gap)
        yield c({}, 'stop')
        yield 'data: [DONE]\n\n'

    return StreamingResponse(gen(), media_type='text/event-stream')


if __name__ == '__main__':
    import uvicorn
    p = argparse.ArgumentParser()
    p.add_argument('--port', type=int, default=9000)
    a = p.parse_args()
    uvicorn.run(app, host='127.0.0.1', port=a.port, log_level='warning')

In [ ]:
%%writefile gateway/scripts/make_replay_set.py
"""RAGTruth dev 응답을 게이트웨이 재생용 JSONL로 만든다.

S1~S2-b와 같은 dev 표본(시드 42, 과제별 120개)에서 과제별 N개를 뽑는다.
각 줄: {id, task_type, reference, response, labels:[{start,end}]}

  python scripts/make_replay_set.py --ragtruth RAGTruth/dataset --per-task 20 --out replay_dev.jsonl
"""
import argparse
import json
import random

p = argparse.ArgumentParser()
p.add_argument('--ragtruth', default='RAGTruth/dataset')
p.add_argument('--per-task', type=int, default=20)
p.add_argument('--out', default='replay_dev.jsonl')
a = p.parse_args()

responses = [json.loads(l) for l in open(f'{a.ragtruth}/response.jsonl')]
sources = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{a.ragtruth}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']


def build_reference(src):          # S2와 동일
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))


TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']
random.seed(42)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)

rows = []
for tt in TASKS:
    for r in [x for x in dev if x['task_type'] == tt][:a.per_task]:
        rows.append({'id': str(r['id']), 'task_type': tt, 'reference': build_reference(sources[r['source_id']]),
                     'response': r['response'],
                     'labels': [{'start': l['start'], 'end': l['end']} for l in (r.get('labels') or [])]})
with open(a.out, 'w') as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(f'{len(rows)}개 응답 → {a.out}')

In [ ]:
%%writefile gateway/scripts/bench.py
"""게이트웨이 종단 간(end-to-end) 측정.

가짜 LLM(mock_upstream)이 RAGTruth dev 응답을 실제 API와 비슷한 속도로 흘려보내고,
게이트웨이를 거쳐 받은 결과를 모드별로 비교한다.

  off  : 검증 없음 (기준선)
  pass : 즉시 전달 + 사후 판정
  hold : 문장 단위 보류 후 판정·전달

측정 항목
  [정확성] 받은 본문이 원래 응답과 글자 단위로 같은지 (게이트웨이가 텍스트를 망가뜨리지 않는지)
  [판별력] 게이트웨이 안에서 나온 문장 점수의 AUROC — 오프라인 실험(S2-b)과 같은 수준인지
  [지연]   첫 글자까지 시간(TTFT), 글자당 평균 지연(off 대비), 판정 소요 시간, pass 모드 판정 도착 지연
  [누출]   pass 모드에서 실제 환각 문장을 판정하기 전까지 사용자에게 이미 나간 글자 수

  python scripts/bench.py --replay replay_dev.jsonl --gateway http://127.0.0.1:8000 \
      --metrics gateway_metrics.jsonl --out bench_out
"""
import argparse
import asyncio
import json
import os
import statistics as st
import time

import httpx


def auroc(y, s):
    pos = [v for v, t in zip(s, y) if t]
    neg = [v for v, t in zip(s, y) if not t]
    if not pos or not neg:
        return float('nan')
    ranked = sorted([(v, i) for i, v in enumerate(s)])
    ranks, i = [0.0] * len(s), 0
    while i < len(ranked):                      # 동점 평균 순위
        j = i
        while j + 1 < len(ranked) and ranked[j + 1][0] == ranked[i][0]:
            j += 1
        for k in range(i, j + 1):
            ranks[ranked[k][1]] = (i + j) / 2 + 1
        i = j + 1
    rp = sum(r for r, t in zip(ranks, y) if t)
    return (rp - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))


def pct(xs, q):
    xs = sorted(x for x in xs if x is not None)
    if not xs:
        return float('nan')
    return xs[min(len(xs) - 1, int(round(q / 100 * (len(xs) - 1))))]


def parse_cfg(cfg):
    """'pass' → ('pass', 0), 'hold:4' → ('hold', 4). 콜론 뒤 숫자는 앞부분 재판정 간격(stride)."""
    m, _, s = cfg.partition(':')
    return m, int(s or 0)


def item_timing(item, a):
    """재생 속도. --timing 파일이 있으면 그 응답의 실측 값(L1: 실제 LLM의 첫 토큰 시간, 글자당 생성 시간)을 쓴다.
    글자당 시간 × (이 응답의 평균 글자/토큰)으로 토큰 간격을 만든다 — 가짜 LLM은 GPT 토큰 단위로 흘려보낸다."""
    t = TIMING.get(item['id'])
    if not t:
        return a.ttft_ms, a.token_ms
    n_tok = len(_ENC.encode(item['response'])) if _ENC else max(1, len(item['response']) / 4.0)
    return t['ttft_ms'], t['ms_per_char'] * len(item['response']) / max(n_tok, 1)


async def one(client, gw, item, mode, a):
    vmode, stride = parse_cfg(mode)
    ttft_ms, token_ms = item_timing(item, a)
    body = {'model': 'mock', 'stream': True,
            'messages': [{'role': 'system', 'content': 'Answer using only the provided context.'},
                         {'role': 'user', 'content': 'Respond based on the context.'}],
            'verification': {'context': item['reference'], 'mode': vmode, 'action': 'flag', 'stride': stride},
            'mock_response': item['response'], 'mock_ttft_ms': ttft_ms, 'mock_token_ms': token_ms}
    tag = f"{item['id']}|{mode}"
    t0 = time.perf_counter()
    timeline, text, verdicts = [], '', []
    async with client.stream('POST', f'{gw}/v1/chat/completions', json=body,
                             headers={'x-request-tag': tag}) as r:
        r.raise_for_status()
        async for line in r.aiter_lines():
            if not line.startswith('data:') or line.endswith('[DONE]'):
                continue
            c = json.loads(line[5:])
            now = (time.perf_counter() - t0) * 1000
            if c.get('verification'):
                verdicts.append({**c['verification'], 't_client': now})
            for ch in c.get('choices') or []:
                d = (ch.get('delta') or {}).get('content')
                if d:
                    text += d
                    timeline.append((now, len(text)))
    return {'id': item['id'], 'task_type': item['task_type'], 'mode': mode, 'tag': tag,
            'text_ok': text == item['response'], 'timeline': timeline, 'verdicts': verdicts,
            't_total': (time.perf_counter() - t0) * 1000}


def onset_metrics(rs, items):
    """환각 발생 시점 기준 지표 (SSOT 결정: 누출은 '환각 발생부터' 센다).

    발생 위치 θ = 응답에서 첫 환각 구간의 시작 글자. 클라이언트가 받은 판정 이벤트를 받은 순서대로 본다.
    - 탐지(엄격): 응답의 첫 경보(hallucinated=true)가 θ를 포함하는 텍스트를 판정한 것(upto > θ)일 때만 탐지.
      첫 경보가 θ 이전 텍스트에 대한 것이면 발생 전 오경보로 끝난 것(미탐지).
    - 탐지 지연(글자): 경보 시점에 판정한 텍스트의 끝 − θ (생성이 θ를 지나 얼마나 진행됐을 때 잡았나)
    - 발생 후 누출(글자): 경보가 나온 순간까지 사용자에게 나간 글자 중 θ 이후 글자 수
      미탐지면 θ 이후 전부가 누출된 것으로 보고 '기대 누출'에 포함
    - 멀쩡한 응답 표시율: 환각 없는 응답 중 경보가 한 번이라도 나간 비율
    """
    info = {it['id']: it for it in items}
    det, delay, leak_det, leak_all, clean_flag = [], [], [], [], []
    for r in rs:
        it = info[r['id']]
        alarms = [v for v in r['verdicts'] if v.get('hallucinated') and 'upto' in v]
        if not it['labels']:
            clean_flag.append(bool(alarms)); continue
        theta = min(l['start'] for l in it['labels'])
        after = len(it['response']) - theta
        first = alarms[0] if alarms else None
        if first is not None and first['upto'] > theta:
            det.append(True)
            delay.append(first['upto'] - theta)
            lk = max(0, first.get('released_at_verdict', 0) - theta)
            leak_det.append(lk); leak_all.append(lk)
        else:
            det.append(False); leak_all.append(after)
    n = max(len(det), 1)
    return {'onset_n_hallucinated': len(det), 'onset_recall_strict': sum(det) / n,
            'onset_detect_delay_chars_p50': pct(delay, 50),
            'onset_leak_chars_p50_detected': pct(leak_det, 50),
            'onset_leak_chars_mean_detected': st.mean(leak_det) if leak_det else float('nan'),
            'onset_expected_leak_chars_mean': st.mean(leak_all) if leak_all else float('nan'),
            'clean_resp_flagged': sum(clean_flag) / max(len(clean_flag), 1)}


def char_times(timeline, n):
    """글자 i가 사용자 화면에 나타난 시각 리스트."""
    out, j = [], 0
    for i in range(n):
        while j < len(timeline) and timeline[j][1] < i + 1:
            j += 1
        out.append(timeline[j][0] if j < len(timeline) else None)
    return out


TIMING = {}
try:
    import tiktoken
    _ENC = tiktoken.get_encoding('cl100k_base')
except Exception:          # 오프라인 등
    _ENC = None


async def main(a):
    items = [json.loads(l) for l in open(a.replay)][:a.n or None]
    if a.timing:
        for l in open(a.timing):
            t = json.loads(l)
            TIMING[str(t['id'])] = t
        print(f'실측 재생 속도 사용: {a.timing} ({len(TIMING)}개 응답)')
    if os.path.exists(a.metrics):
        os.remove(a.metrics)                      # 이번 측정분만 남기기 (게이트웨이와 같은 경로여야 함)
    runs = []
    async with httpx.AsyncClient(timeout=300) as client:
        for i, it in enumerate(items):
            for mode in a.modes:
                runs.append(await one(client, a.gateway, it, mode, a))
            if (i + 1) % 10 == 0:
                print(f'  {i + 1}/{len(items)} 응답 완료')
    await asyncio.sleep(0.5)
    recs = {}
    if os.path.exists(a.metrics):
        for l in open(a.metrics):
            r = json.loads(l)
            recs[r['tag']] = r

    by = {(r['id'], r['mode']): r for r in runs}
    labels = {it['id']: it['labels'] for it in items}
    lens = {it['id']: len(it['response']) for it in items}
    os.makedirs(a.out, exist_ok=True)
    summary, sent_rows = {}, []

    for mode in a.modes:
        rs = [r for r in runs if r['mode'] == mode]
        s = {'n_responses': len(rs), 'text_ok_rate': sum(r['text_ok'] for r in rs) / len(rs),
             'ttft_ms_p50': pct([r['timeline'][0][0] if r['timeline'] else None for r in rs], 50),
             'total_ms_p50': pct([r['t_total'] for r in rs], 50)}
        if mode != 'off' and 'off' in a.modes:
            delays = []
            for r in rs:
                off = by[(r['id'], 'off')]
                n = lens[r['id']]
                ct, co = char_times(r['timeline'], n), char_times(off['timeline'], n)
                delays += [x - y for x, y in zip(ct, co) if x is not None and y is not None]
            s['char_delay_ms_mean'] = st.mean(delays) if delays else float('nan')
            s['char_delay_ms_p95'] = pct(delays, 95)
            s['ttft_added_ms_p50'] = pct([(r['timeline'][0][0] - by[(r['id'], 'off')]['timeline'][0][0])
                                          for r in rs if r['timeline']], 50)
        if mode != 'off':
            rows = []
            prep = []
            for r in rs:
                rec = recs.get(r['tag'])
                if not rec:
                    continue
                prep.append(rec['prepare_ms'])
                for sr in rec['sentences']:
                    if sr.get('skipped') or 'score' not in sr:
                        continue
                    a0, a1 = sr['start'], sr['start'] + sr['n_chars']
                    y = int(any(min(a1, l['end']) > max(a0, l['start']) for l in labels[r['id']]))
                    rows.append({'id': r['id'], 'task_type': r['task_type'], 'mode': mode,
                                 'sentence_idx': sr['sentence_idx'], 'y': y, 'score': sr['score'],
                                 'hallucinated': sr['hallucinated'], 'judge_ms': sr['judge_ms'],
                                 'verdict_lag_ms': sr['t_verdict'] - sr['t_complete'],
                                 'release_delay_ms': (sr['t_release'] - sr['t_complete'])
                                 if sr.get('t_release') is not None else None,
                                 'leak_chars': sr.get('leak_chars')})
            sent_rows += rows
            ys = [x['y'] for x in rows]
            tp = sum(1 for x in rows if x['y'] and x['hallucinated'])
            s.update({'n_sentences': len(rows), 'pos_rate': sum(ys) / max(len(ys), 1),
                      'AUROC': auroc(ys, [x['score'] for x in rows]),
                      'recall_at_threshold': tp / max(sum(ys), 1),
                      'precision_at_threshold': tp / max(sum(x['hallucinated'] for x in rows), 1),
                      'judge_ms_p50': pct([x['judge_ms'] for x in rows], 50),
                      'judge_ms_p95': pct([x['judge_ms'] for x in rows], 95),
                      'verdict_lag_ms_p50': pct([x['verdict_lag_ms'] for x in rows], 50),
                      'verdict_lag_ms_p95': pct([x['verdict_lag_ms'] for x in rows], 95),
                      'prepare_ms_p50': pct(prep, 50)})
            if parse_cfg(mode)[0] == 'pass':
                # (구 지표) 실제 환각이면서 잡아낸 문장: 판정이 나올 때까지 그 문장 시작부터 사용자에게 나간 글자 수
                caught = [x['leak_chars'] for x in rows if x['y'] and x['hallucinated']]
                s['leak_chars_p50_on_caught'] = pct(caught, 50)
                s['leak_chars_p95_on_caught'] = pct(caught, 95)
            s.update(onset_metrics(rs, items))
        summary[mode] = s

    json.dump(summary, open(f'{a.out}/bench_summary.json', 'w'), indent=1, ensure_ascii=False)
    with open(f'{a.out}/bench_sentences.jsonl', 'w') as f:
        for row in sent_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    speed = f'실측 속도 {a.timing}' if a.timing else f'토큰 간격 {a.token_ms}ms, 첫 토큰 {a.ttft_ms}ms'
    print(f'\n=== 결과 ({len(items)}개 응답, {speed}) ===')
    for mode, s in summary.items():
        print(f'\n[{mode}]')
        for k, v in s.items():
            print(f'  {k:26} {v:.3f}' if isinstance(v, float) else f'  {k:26} {v}')
    print(f'\n저장: {a.out}/bench_summary.json, {a.out}/bench_sentences.jsonl')


if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--replay', default='replay_dev.jsonl')
    p.add_argument('--gateway', default='http://127.0.0.1:8000')
    p.add_argument('--metrics', default='gateway_metrics.jsonl', help='게이트웨이의 METRICS_PATH와 같은 파일')
    p.add_argument('--modes', nargs='+', default=['off', 'pass', 'hold'],
                   help="검증 설정 목록. 'pass:4'처럼 콜론 뒤에 앞부분 재판정 간격(stride)을 붙일 수 있다")
    p.add_argument('--n', type=int, default=0, help='앞에서 n개만 (0=전체)')
    p.add_argument('--ttft-ms', type=float, default=300)
    p.add_argument('--token-ms', type=float, default=25)
    p.add_argument('--timing', default='', help='L1 실측 재생 속도 파일 (id, ttft_ms, ms_per_char). 없으면 고정값')
    p.add_argument('--out', default='bench_out')
    asyncio.run(main(p.parse_args()))

In [ ]:
%%writefile gateway/tests/test_gateway.py
"""게이트웨이 단위·통합 테스트 (모델 다운로드 없이 실행 가능).

  cd gateway && python -m pytest -q
"""
import json
import os
import random
import sys

import httpx
import pytest

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from hallugate import Settings, create_app                      # noqa: E402
from hallugate.context import extract                            # noqa: E402
from hallugate.judge import LexicalJudge                         # noqa: E402
from hallugate.splitter import IncrementalSplitter               # noqa: E402
import mock_upstream                                             # noqa: E402

REF = 'The company was founded in 1998 in Seoul. It reported revenue of 2 billion dollars in 2023.'
GOOD = 'The company was founded in 1998 in Seoul. It reported revenue of 2 billion dollars in 2023.'
BAD_2ND = 'The company was founded in 1998 in Seoul. Its chief executive flew rockets across Mars yesterday. It reported revenue of 2 billion dollars in 2023.'


# ---------------- 문장 분할 ----------------
@pytest.mark.parametrize('text', [
    GOOD, BAD_2ND,
    'Dr. Kim arrived at 3.5 p.m. on Jan. 5. He left early!  Then what?\n\nNew paragraph here.',
    'No terminal punctuation at all',
])
def test_splitter_streaming_matches_whole(text):
    rng = random.Random(0)
    sp, got, i = IncrementalSplitter(), [], 0
    while i < len(text):
        j = i + rng.randint(1, 6)
        got += sp.feed(text[i:j])
        i = j
    got += sp.flush()
    assert ''.join(s.text for s in got) == text                    # 글자 하나도 잃지 않음
    assert [s.start for s in got] == [sum(len(x.text) for x in got[:k]) for k in range(len(got))]
    whole = IncrementalSplitter()
    ref = whole.feed(text) + whole.flush()
    assert [s.clean for s in got] == [s.clean for s in ref]        # 조각 단위로 받아도 한 번에 받은 것과 같음


# ---------------- 컨텍스트 추출 ----------------
def test_extract_field_and_strip():
    body = {'model': 'x', 'messages': [], 'verification': {'context': ['a', 'b'], 'mode': 'hold', 'action': 'cut'}}
    up, ref, mode, action = extract(body)
    assert 'verification' not in up and ref == 'a\n\nb' and (mode, action) == ('hold', 'cut')


def test_extract_tag_and_off():
    body = {'messages': [{'role': 'user', 'content': 'Q?\n<context>doc one</context> <context>doc two</context>'}]}
    assert extract(body)[1:3] == ('doc one\n\ndoc two', 'pass')
    assert extract({'messages': [{'role': 'user', 'content': 'hi'}]})[2] == 'off'
    with pytest.raises(ValueError):
        extract({'messages': [], 'verification': {'context': 'x', 'mode': 'bogus'}})


# ---------------- 게이트웨이 통합 (가짜 LLM 연결) ----------------
@pytest.fixture
def gw(tmp_path):
    up = httpx.AsyncClient(transport=httpx.ASGITransport(app=mock_upstream.app), base_url='http://mock')
    s = Settings(upstream_base_url='http://mock/v1', metrics_path=str(tmp_path / 'm.jsonl'))
    app = create_app(s, judge=LexicalJudge(threshold=0.5), http_client=up)
    return httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url='http://gw'), s


def body(resp, mode, action='flag', stream=True):
    return {'model': 'mock', 'stream': stream, 'messages': [{'role': 'user', 'content': 'q'}],
            'verification': {'context': REF, 'mode': mode, 'action': action},
            'mock_response': resp, 'mock_ttft_ms': 0, 'mock_token_ms': 0}


async def run(client, b):
    r = await client.post('/v1/chat/completions', json=b)
    assert r.status_code == 200
    text, events, finish = '', [], None
    for line in r.text.splitlines():
        if not line.startswith('data:') or line.endswith('[DONE]'):
            continue
        c = json.loads(line[5:])
        if c.get('verification'):
            events.append(('v', c['verification']))
        for ch in c['choices']:
            d = ch['delta'].get('content')
            if d:
                text += d
                events.append(('t', d))
            finish = ch.get('finish_reason') or finish
    return text, events, finish


@pytest.mark.asyncio
@pytest.mark.parametrize('mode', ['pass', 'hold'])
async def test_stream_text_intact_and_verdicts(gw, mode):
    client, s = gw
    text, ev, finish = await run(client, body(BAD_2ND, mode))
    assert text == BAD_2ND and finish == 'stop'
    vs = [e for k, e in ev if k == 'v']
    assert [v['sentence_idx'] for v in vs] == [0, 1, 2]
    assert [v['hallucinated'] for v in vs] == [False, True, False]
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    assert rec['mode'] == mode and len(rec['sentences']) == 3
    if mode == 'hold':
        # hold: 각 문장의 판정이 그 문장 본문보다 먼저 나가야 함 → 누출 0
        order = [k for k, _ in ev]
        assert order == ['v', 't', 'v', 't', 'v', 't']
        assert all(x.get('leak_chars', 0) == 0 for x in rec['sentences'])


@pytest.mark.asyncio
async def test_hold_cut_blocks_bad_sentence(gw):
    client, _ = gw
    text, ev, finish = await run(client, body(BAD_2ND, 'hold', 'cut'))
    assert finish == 'content_filter'
    assert text == 'The company was founded in 1998 in Seoul. '     # 틀린 문장은 한 글자도 안 나감
    assert 'rockets' not in text


# ---------------- 실제 서버 (시간 간격이 있는 스트리밍) ----------------
# ASGITransport는 응답을 한 번에 모아서 주므로, 토큰 간격이 중요한 테스트는 실제 서버로 한다.
def _free_port():
    import socket
    s = socket.socket(); s.bind(('127.0.0.1', 0)); p = s.getsockname()[1]; s.close(); return p


@pytest.fixture
def live(tmp_path):
    import threading, time as _t, uvicorn
    servers = []

    def start(app):
        port = _free_port()
        srv = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=port, log_level='error'))
        th = threading.Thread(target=srv.run, daemon=True); th.start()
        while not srv.started:
            _t.sleep(0.02)
        servers.append(srv)
        return f'http://127.0.0.1:{port}'

    up = start(mock_upstream.app)
    s = Settings(upstream_base_url=up + '/v1', metrics_path=str(tmp_path / 'm.jsonl'))
    gw_url = start(create_app(s, judge=LexicalJudge(threshold=0.5, delay_ms=5)))
    yield gw_url, s
    for srv in servers:
        srv.should_exit = True


async def run_live(url, b):
    """(시각ms, 종류, 값) 이벤트 목록"""
    import time as _t
    out, t0 = [], _t.perf_counter()
    async with httpx.AsyncClient(timeout=30) as c:
        async with c.stream('POST', url + '/v1/chat/completions', json=b) as r:
            async for line in r.aiter_lines():
                if not line.startswith('data:') or line.endswith('[DONE]'):
                    continue
                ch = json.loads(line[5:]); now = (_t.perf_counter() - t0) * 1000
                if ch.get('verification'):
                    out.append((now, 'v', ch['verification']))
                for x in ch['choices']:
                    if x['delta'].get('content'):
                        out.append((now, 't', x['delta']['content']))
                    if x.get('finish_reason'):
                        out.append((now, 'f', x['finish_reason']))
    return out


@pytest.mark.asyncio
async def test_live_pass_cut_stops_after_verdict(live):
    url, _ = live
    b = body(BAD_2ND, 'pass', 'cut'); b['mock_token_ms'] = 20
    ev = await run_live(url, b)
    text = ''.join(v for _, k, v in ev if k == 't')
    assert ('f', 'content_filter') in [(k, v) for _, k, v in ev]
    assert 'rockets' in text and '2023' not in text        # 이미 나간 건 못 막지만 이후는 끊김


@pytest.mark.asyncio
async def test_live_pass_verdict_arrives_without_waiting_next_token(live):
    url, s = live
    b = body(BAD_2ND, 'pass'); b['mock_token_ms'] = 50
    ev = await run_live(url, b)
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    lags = [x['t_verdict'] - x['t_complete'] for x in rec['sentences']]
    assert max(lags) < 40, lags                              # 다음 토큰(50ms)을 기다리지 않고 바로 판정 전달


@pytest.mark.asyncio
async def test_live_hold_delays_but_no_leak(live):
    url, s = live
    for mode in ['off', 'hold']:
        b = body(BAD_2ND, mode); b['mock_token_ms'] = 20
        if mode == 'off':
            b.pop('verification')
        ev = await run_live(url, b)
        text = ''.join(v for _, k, v in ev if k == 't')
        assert text == BAD_2ND
        first = next(t for t, k, _ in ev if k == 't')
        if mode == 'off':
            off_first = first
        else:
            assert first > off_first + 100                   # 첫 문장이 다 만들어질 때까지 보류됨


@pytest.mark.asyncio
async def test_off_is_passthrough(gw):
    client, _ = gw
    b = body(GOOD, 'pass')
    b.pop('verification')
    r = await client.post('/v1/chat/completions', json=b)
    assert r.headers['x-verification-mode'] == 'off'
    assert 'verification' not in r.text


@pytest.mark.asyncio
async def test_non_stream(gw):
    client, _ = gw
    r = await client.post('/v1/chat/completions', json=body(BAD_2ND, 'hold', 'cut', stream=False))
    d = r.json()
    assert d['choices'][0]['finish_reason'] == 'content_filter'
    assert d['choices'][0]['message']['content'] == 'The company was founded in 1998 in Seoul. '
    assert [x['hallucinated'] for x in d['verification']['sentences']] == [False, True]


@pytest.mark.asyncio
async def test_openai_sdk_compatible(gw):
    from openai import AsyncOpenAI
    client, _ = gw
    oai = AsyncOpenAI(base_url='http://gw/v1', api_key='x', http_client=client)
    stream = await oai.chat.completions.create(
        model='mock', messages=[{'role': 'user', 'content': 'q'}], stream=True,
        extra_body={'verification': {'context': REF, 'mode': 'hold'}, 'mock_response': BAD_2ND,
                    'mock_ttft_ms': 0, 'mock_token_ms': 0})
    text, flags = '', []
    async for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            text += chunk.choices[0].delta.content
        v = getattr(chunk, 'verification', None)
        if v:
            flags.append(v['hallucinated'])
    assert text == BAD_2ND and flags == [False, True, False]


# ---------------- 학습된 판정기 로딩 경로 (작은 임의 모델) ----------------
def test_student_judge_loads(tmp_path):
    import torch
    from transformers import BertConfig, BertForSequenceClassification, BertTokenizerFast
    from hallugate.judge import StudentJudge
    words = 'the company was founded in seoul revenue billion dollars reported it'.split()
    (tmp_path / 'vocab.txt').write_text('\n'.join(['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '...', '.'] + words))
    BertTokenizerFast(str(tmp_path / 'vocab.txt')).save_pretrained(tmp_path)
    BertForSequenceClassification(BertConfig(vocab_size=len(words) + 7, hidden_size=16, num_hidden_layers=1,
                                             num_attention_heads=2, intermediate_size=32, num_labels=1)
                                  ).save_pretrained(tmp_path)

    class Emb:
        def encode(self, xs, **k):
            return torch.nn.functional.normalize(torch.randn(len(xs), 8), dim=-1)

    j = StudentJudge(str(tmp_path), embedder=Emb(), k=8, device='cpu')
    ref = j.prepare(REF * 5)
    v = j.judge(ref, 'The company was founded in 1998.')
    assert isinstance(v.score, float) and v.judge_ms > 0


# ---------------- 앞부분 재판정 (stride, SPEC §5.1) ----------------
def test_splitter_pending_tracks_current_sentence():
    sp = IncrementalSplitter()
    sp.feed('First sentence here. Second sen')
    off, buf = sp.pending()
    assert off == len('First sentence here. ') and buf == 'Second sen'


def test_extract_stride():
    body = {'messages': [], 'verification': {'context': 'x', 'stride': 4}}
    assert extract(body, with_stride=True)[4] == 4
    assert extract({'messages': [], 'verification': {'context': 'x'}}, default_stride=2, with_stride=True)[4] == 2
    with pytest.raises(ValueError):
        extract({'messages': [], 'verification': {'context': 'x', 'stride': -1}}, with_stride=True)


@pytest.mark.asyncio
@pytest.mark.parametrize('mode', ['pass', 'hold'])
async def test_stride_text_intact_and_prefix_verdicts(gw, mode):
    client, s = gw
    b = body(BAD_2ND, mode); b['verification']['stride'] = 2
    text, ev, finish = await run(client, b)
    assert text == BAD_2ND and finish == 'stop'                    # 본문 보존 (G2)
    vs = [e for k, e in ev if k == 'v']
    assert {'prefix', 'sentence'} <= {v['kind'] for v in vs}
    assert all(v['start'] < v['upto'] and 'released_at_verdict' in v for v in vs)
    # 두 번째 문장은 앞부분 판정에서 이미 환각으로 잡힘 (문장 판정보다 먼저)
    first_alarm = next(v for v in vs if v['hallucinated'])
    assert first_alarm['kind'] == 'prefix' and first_alarm['sentence_idx'] == 1
    rec = json.loads(open(s.metrics_path).read().splitlines()[-1])
    assert rec['stride'] == 2 and len(rec['checks']) > 0


@pytest.mark.asyncio
async def test_stride_hold_releases_safe_prefix_before_sentence_end(gw):
    client, _ = gw
    b = body(GOOD, 'hold'); b['verification']['stride'] = 2
    text, ev, _ = await run(client, b)
    assert text == GOOD
    # 첫 문장의 문장 판정('sentence') 이전에, 안전한 앞부분이 이미 전달돼야 한다
    first_sent_verdict = next(i for i, (k, e) in enumerate(ev) if k == 'v' and e['kind'] == 'sentence')
    texts_before = [e for k, e in ev[:first_sent_verdict] if k == 't']
    assert texts_before, '문장 판정 전에 아무것도 전달되지 않음'
    # 첫 전달은 문장 전체가 아니라 앞부분이어야 한다 (문장 끝까지 기다리지 않음)
    assert len(texts_before[0]) < len('The company was founded in 1998 in Seoul. ')


@pytest.mark.asyncio
async def test_stride_hold_cut_stops_inside_bad_sentence(gw):
    client, _ = gw
    b = body(BAD_2ND, 'hold', 'cut'); b['verification']['stride'] = 2
    text, ev, finish = await run(client, b)
    assert finish == 'content_filter'
    assert text == 'The company was founded in 1998 in Seoul. '     # 환각 문장은 한 글자도 나가지 않음
    cut = [e for k, e in ev if k == 'v' and e.get('cut')]
    assert cut and cut[0]['kind'] == 'prefix'


@pytest.mark.asyncio
async def test_live_stride_hold_reduces_wait(live):
    # 실제 서버: 앞부분 재판정을 켜면 hold의 첫 글자 지연이 문장 길이보다 짧아진다
    url, _ = live
    firsts = {}
    for stride in [0, 2]:
        b = body(BAD_2ND, 'hold'); b['mock_token_ms'] = 20; b['verification']['stride'] = stride
        ev = await run_live(url, b)
        assert ''.join(v for _, k, v in ev if k == 't') == BAD_2ND
        firsts[stride] = next(t for t, k, _ in ev if k == 't')
    assert firsts[2] < firsts[0] - 60, firsts

In [ ]:
%%writefile gateway/pytest.ini
[pytest]
asyncio_mode = strict

In [ ]:
!cd gateway && python -m pytest -q 2>&1 | tail -3

## 3. 재생 데이터 — RAGTruth dev 과제별 20개 (S3와 같음)

In [ ]:
!cd gateway && python scripts/make_replay_set.py --ragtruth ../RAGTruth/dataset --per-task 20 --out replay_dev.jsonl

## 4. 가짜 LLM(GPT 토큰 단위)과 게이트웨이 3개 실행

In [ ]:
import subprocess, time, httpx

def start(cmd, env_extra, log):
    return subprocess.Popen(cmd, cwd='gateway', env={**os.environ, **env_extra},
                            stdout=open(log, 'w'), stderr=subprocess.STDOUT)

def wait(url, proc, log, sec=180):
    for _ in range(sec):
        try:
            return httpx.get(url, timeout=2).json()
        except Exception:
            if proc.poll() is not None: break
            time.sleep(1)
    print(open(log).read()[-3000:]); raise RuntimeError(f'{url} 시작 실패')

mock = start(['python', 'mock_upstream.py', '--port', '9000'], {'MOCK_TOKENIZER': 'tiktoken'}, 'mock.log')
GW = {'B': (8001, '../student_vark'), 'E-end': (8002, '../student_prefix'), 'E-s4': (8003, '../student_prefix')}
procs = {}
for name, (port, model) in GW.items():
    procs[name] = start(['python', 'serve.py', '--port', str(port)],
                        {'JUDGE_MODEL_DIR': model, 'UPSTREAM_BASE_URL': 'http://127.0.0.1:9000/v1',
                         'METRICS_PATH': f'metrics_{name}.jsonl', 'JUDGE_THRESHOLD': str(THR[name]),
                         'JUDGE_K': '8', 'JUDGE_MAX_LEN': '512'}, f'gw_{name}.log')
    time.sleep(2)
for name, (port, _) in GW.items():
    print(name, wait(f'http://127.0.0.1:{port}/health', procs[name], f'gw_{name}.log'))

## 5. 종단 간 측정 (구성마다 off·pass·hold, 약 35분)

In [ ]:
MODES = {'B': 'off pass hold', 'E-end': 'off pass hold', 'E-s4': 'off pass:4 hold:4'}
for name, (port, _) in GW.items():
    print(f'=== {name} ===')
    !cd gateway && python scripts/bench.py --replay replay_dev.jsonl --gateway http://127.0.0.1:{port} \
        --metrics metrics_{name}.jsonl --modes {MODES[name]} --out bench_{name} | tail -2

## 6. 결과

In [ ]:
rows = []
for name in GW:
    S = json.load(open(f'gateway/bench_{name}/bench_summary.json'))
    for mode, s in S.items():
        if mode == 'off': continue
        rows.append({'config': name, 'mode': mode.split(':')[0], **s})
R = pd.DataFrame(rows)
cols = ['config', 'mode', 'text_ok_rate', 'ttft_added_ms_p50', 'char_delay_ms_mean', 'char_delay_ms_p95',
        'judge_ms_p95', 'onset_n_hallucinated', 'onset_recall_strict', 'onset_detect_delay_chars_p50',
        'onset_leak_chars_p50_detected', 'onset_expected_leak_chars_mean', 'clean_resp_flagged', 'AUROC']
pd.set_option('display.width', 250)
print(R[cols].round(3).to_string(index=False))

## 7. 사전 기준 판정

In [ ]:
g = R.set_index(['config', 'mode'])
b_hold, e_hold = g.loc[('B', 'hold')], g.loc[('E-s4', 'hold')]
b_pass, e_pass = g.loc[('B', 'pass')], g.loc[('E-s4', 'pass')]
c1 = e_hold.ttft_added_ms_p50 <= 0.5 * b_hold.ttft_added_ms_p50
c2 = e_pass.onset_leak_chars_p50_detected <= 0.5 * b_pass.onset_leak_chars_p50_detected
c3 = e_pass.onset_recall_strict >= b_pass.onset_recall_strict - 0.05
c4 = (R.text_ok_rate == 1.0).all() and (R.judge_ms_p95 <= 30).all()
print(f"C1 대기: hold 첫 글자 추가 지연 B {b_hold.ttft_added_ms_p50:.0f}ms → E-s4 {e_hold.ttft_added_ms_p50:.0f}ms "
      f"(글자당 평균 {b_hold.char_delay_ms_mean:.0f} → {e_hold.char_delay_ms_mean:.0f}ms) → {'충족' if c1 else '미달'}")
print(f"C2 누출: pass 발생 후 누출 중앙값 B {b_pass.onset_leak_chars_p50_detected:.0f}자 → E-s4 "
      f"{e_pass.onset_leak_chars_p50_detected:.0f}자 → {'충족' if c2 else '미달'}")
print(f"C3 탐지: 엄격 재현율 B {b_pass.onset_recall_strict:.3f} → E-s4 {e_pass.onset_recall_strict:.3f} "
      f"(환각 응답 {int(e_pass.onset_n_hallucinated)}개) → {'충족' if c3 else '미달'}")
print(f"C4 기본: 본문 보존 {R.text_ok_rate.min():.0%}, 판정 p95 최대 {R.judge_ms_p95.max():.1f}ms → {'충족' if c4 else '미달'}")
print(f"\n참고 — 멀쩡한 응답 표시율: " + ', '.join(f'{c}/{m} {v:.0%}' for (c, m), v in g.clean_resp_flagged.items()))

In [ ]:
!cd gateway && zip -qr ../e5lite_results.zip bench_B bench_E-end bench_E-s4 metrics_B.jsonl metrics_E-end.jsonl metrics_E-s4.jsonl
shutil.copy('e5lite_results.zip', os.path.join(OUT_DIR, 'e5lite_results.zip'))
from google.colab import files
files.download('e5lite_results.zip')
print('드라이브에도 저장:', os.path.join(OUT_DIR, 'e5lite_results.zip'))

## 8. (L1 이후) 실제 LLM 속도로 다시 측정

L1 노트북을 먼저 실행해 드라이브 `selfproject/l1/timing_<모델>.jsonl`이 있으면, 가짜 LLM이 RAGTruth 응답을
**그 모델의 실제 첫 토큰 시간과 생성 속도**로 흘려보내도록 해서 B와 E-s4를 다시 비교한다.
시간을 줄이기 위해 모델마다 30개 응답만 쓴다(모델당 약 15분). 측정할 모델은 `TIMING_MODELS`에서 고른다.

In [ ]:
import glob
L1 = os.path.join(OUT_DIR, 'l1')
avail = sorted(os.path.basename(f)[7:-6] for f in glob.glob(os.path.join(L1, 'timing_*.jsonl')))
print('실측 속도 파일:', avail)
TIMING_MODELS = avail                      # 일부만 하려면 예: ['claude-haiku-4-5-20251001', 'gpt-5.4-mini']
N_REAL = 30
for tm in TIMING_MODELS:
    shutil.copy(os.path.join(L1, f'timing_{tm}.jsonl'), f'gateway/timing_{tm}.jsonl')
    for name, modes in [('B', 'off pass hold'), ('E-s4', 'off pass:4 hold:4')]:
        port = GW[name][0]
        print(f'=== {tm} | {name} ===')
        !cd gateway && python scripts/bench.py --replay replay_dev.jsonl --gateway http://127.0.0.1:{port} --n {N_REAL} \
            --metrics metrics_{name}.jsonl --modes {modes} --timing timing_{tm}.jsonl --out real_{tm}_{name} | tail -1

In [ ]:
rows = []
for tm in TIMING_MODELS:
    for name in ['B', 'E-s4']:
        S = json.load(open(f'gateway/real_{tm}_{name}/bench_summary.json'))
        for mode, s in S.items():
            rows.append({'llm': tm, 'config': name, 'mode': mode.split(':')[0], 'total_ms_p50': s['total_ms_p50'],
                         'ttft_ms_p50': s['ttft_ms_p50'], 'ttft_added_ms_p50': s.get('ttft_added_ms_p50'),
                         'char_delay_ms_mean': s.get('char_delay_ms_mean'),
                         'onset_leak_chars_p50_detected': s.get('onset_leak_chars_p50_detected'),
                         'onset_recall_strict': s.get('onset_recall_strict')})
RR = pd.DataFrame(rows)
RR['added_vs_total'] = RR.ttft_added_ms_p50 / RR.total_ms_p50
print(RR.round(3).to_string(index=False))
RR.to_csv(os.path.join(OUT_DIR, 'e5lite_real_timing.csv'), index=False)
!cd gateway && zip -qr ../e5lite_real_timing.zip real_*
shutil.copy('e5lite_real_timing.zip', os.path.join(OUT_DIR, 'e5lite_real_timing.zip'))
print('저장:', os.path.join(OUT_DIR, 'e5lite_real_timing.csv'))